In [ ]:
import os
import sys
import joblib
import logging
import numpy as np
import pandas as pd
import rasterio
import lightgbm as lgb

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report

In [ ]:
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    handlers=[logging.StreamHandler(sys.stdout)],
    force=True
)

logging.info("Logging is working ✅")

In [ ]:
BASE_DIR = "/content/drive/MyDrive/PhytoLB"

GBIF_PATH = f"{BASE_DIR}/dwca-tree_lebanon-v1.0/occurrence.txt"
CLIMATE_DIR = f"{BASE_DIR}/wc2.1_10m_bio"
ELEVATION_PATH = f"{BASE_DIR}/elevation/elevation.tif"

SAVE_DIR = f"{BASE_DIR}/model_enhanced_v4"
os.makedirs(SAVE_DIR, exist_ok=True)

logging.info(f"Saving outputs to: {SAVE_DIR}")

In [ ]:
def load_gbif_txt(path):
    df = pd.read_csv(path, sep="\t", low_memory=False)

    if "species" not in df.columns:
        df["species"] = df["scientificName"]

    df = df[["species", "decimalLatitude", "decimalLongitude"]]
    df = df.dropna()

    df.columns = ["species", "lat", "lon"]

    df["lat"] = pd.to_numeric(df["lat"], errors="coerce")
    df["lon"] = pd.to_numeric(df["lon"], errors="coerce")
    df = df.dropna()

    # Lebanon bounding box
    df = df[
        (df["lat"] >= 33.0) & (df["lat"] <= 34.8) &
        (df["lon"] >= 35.0) & (df["lon"] <= 36.8)
    ]

    df = df.drop_duplicates()

    logging.info(f"GBIF loaded: {len(df)} records")
    logging.info(f"Unique species: {df['species'].nunique()}")

    return df

In [ ]:
class EnvironmentalExtractor:
    def __init__(self, climate_folder, elevation_path):
        self.climate_rasters = []

        for i in range(1, 20):
            path = os.path.join(climate_folder, f"wc2.1_10m_bio_{i}.tif")

            if not os.path.exists(path):
                raise FileNotFoundError(f"Missing climate file: {path}")

            self.climate_rasters.append(rasterio.open(path))

        if not os.path.exists(elevation_path):
            raise FileNotFoundError(f"Missing elevation file: {elevation_path}")

        self.elevation_raster = rasterio.open(elevation_path)

        logging.info("Climate + elevation rasters loaded ✅")

    def extract_point(self, lat, lon):
        values = []

        for raster in self.climate_rasters:
            try:
                val = next(raster.sample([(lon, lat)]))[0]

                if raster.nodata is not None and val == raster.nodata:
                    val = np.nan

            except Exception:
                val = np.nan

            values.append(val)

        try:
            elev = next(self.elevation_raster.sample([(lon, lat)]))[0]

            if self.elevation_raster.nodata is not None and elev == self.elevation_raster.nodata:
                elev = np.nan

        except Exception:
            elev = np.nan

        values.append(elev)

        return values

    def extract_dataframe(self, df):
        features = []
        total = len(df)

        for i, row in enumerate(df.itertuples(index=False)):
            values = self.extract_point(row.lat, row.lon)
            features.append(values)

            if i % 1000 == 0:
                logging.info(f"Processed {i}/{total} points")

        columns = [f"BIO{i}" for i in range(1, 20)] + ["elevation"]

        return pd.DataFrame(features, columns=columns)

In [ ]:
def generate_pseudo_absence(df, n_samples, extractor, random_state=42):
    np.random.seed(random_state)

    lat_min, lat_max = df["lat"].min(), df["lat"].max()
    lon_min, lon_max = df["lon"].min(), df["lon"].max()

    pseudo_rows = []
    attempts = 0
    max_attempts = n_samples * 10

    logging.info("Generating filtered pseudo-absence points...")

    while len(pseudo_rows) < n_samples and attempts < max_attempts:
        attempts += 1

        lat = np.random.uniform(lat_min, lat_max)
        lon = np.random.uniform(lon_min, lon_max)

        # Extract environmental values immediately
        values = extractor.extract_point(lat, lon)

        # Reject points with too many missing values
        if np.isnan(values).sum() > 2:
            continue

        # Reject invalid elevation values
        elevation = values[-1]

        if np.isnan(elevation):
            continue

        if elevation < -10 or elevation > 3200:
            continue

        species = df["species"].sample(1, random_state=np.random.randint(0, 999999)).iloc[0]

        row = {
            "species": species,
            "lat": lat,
            "lon": lon,
            "label": 0
        }

        pseudo_rows.append(row)

        if len(pseudo_rows) % 1000 == 0:
            logging.info(f"Generated {len(pseudo_rows)}/{n_samples} valid pseudo-absences")

    pseudo_df = pd.DataFrame(pseudo_rows)

    if len(pseudo_df) < n_samples:
        logging.warning(
            f"Only generated {len(pseudo_df)} pseudo-absences out of {n_samples}"
        )

    return pseudo_df

In [ ]:
def add_engineered_features(df):
    df = df.copy()

    # Temperature range / seasonality signals
    df["temp_annual_range"] = df["BIO5"] - df["BIO6"]
    df["warm_cold_quarter_diff"] = df["BIO10"] - df["BIO11"]

    # Precipitation seasonality signals
    df["precip_wet_dry_month_diff"] = df["BIO13"] - df["BIO14"]
    df["precip_wet_dry_quarter_diff"] = df["BIO16"] - df["BIO17"]

    # Ratios with safe division
    df["dryness_ratio"] = df["BIO17"] / (df["BIO16"] + 1e-6)
    df["summer_winter_precip_ratio"] = df["BIO18"] / (df["BIO19"] + 1e-6)

    # Elevation-climate interactions
    df["elevation_temp_interaction"] = df["elevation"] * df["BIO1"]
    df["elevation_precip_interaction"] = df["elevation"] * df["BIO12"]

    return df


In [ ]:
def build_dataset(df, extractor):
    logging.info("Extracting presence features...")

    presence_features = extractor.extract_dataframe(df)
    presence_features["species"] = df["species"].values
    presence_features["label"] = 1

    pseudo_df = generate_pseudo_absence(
        df=df,
        n_samples=len(df),
        extractor=extractor,
        random_state=42
    )

    logging.info("Extracting pseudo-absence features...")

    absence_features = extractor.extract_dataframe(pseudo_df)
    absence_features["species"] = pseudo_df["species"].values
    absence_features["label"] = 0

    full_df = pd.concat(
        [presence_features, absence_features],
        ignore_index=True
    )

    logging.info(f"Dataset before filtering: {full_df.shape}")

    # Remove rows where many environmental features are missing
    env_cols = [col for col in full_df.columns if col.startswith("BIO")] + ["elevation"]
    full_df["missing_count"] = full_df[env_cols].isna().sum(axis=1)

    full_df = full_df[full_df["missing_count"] <= 2]
    full_df = full_df.drop(columns=["missing_count"])

    logging.info(f"Dataset after filtering: {full_df.shape}")

    full_df = add_engineered_features(full_df)

    return full_df

In [ ]:
def get_model():
    return lgb.LGBMClassifier(
        objective="binary",
        n_estimators=900,
        learning_rate=0.02,
        max_depth=9,
        num_leaves=80,
        min_child_samples=15,
        subsample=0.9,
        subsample_freq=1,
        colsample_bytree=0.9,
        reg_alpha=0.05,
        reg_lambda=0.3,
        random_state=42,
        force_row_wise=True,
        verbose=-1
    )

In [ ]:

gbif = load_gbif_txt(GBIF_PATH)
gbif.to_csv(f"{SAVE_DIR}/clean_gbif.csv", index=False)


extractor = EnvironmentalExtractor(
    climate_folder=CLIMATE_DIR,
    elevation_path=ELEVATION_PATH
)

dataset = build_dataset(gbif, extractor)

logging.info(f"Final dataset shape: {dataset.shape}")
logging.info(f"Class distribution:\n{dataset['label'].value_counts()}")
logging.info(f"Missing values:\n{dataset.isna().sum().sort_values(ascending=False).head(10)}")

dataset.to_csv(f"{SAVE_DIR}/final_phyto_dataset.csv", index=False)
dataset.to_parquet(f"{SAVE_DIR}/final_phyto_dataset.parquet", index=False)

logging.info("Final dataset saved ✅")


X = dataset.drop(columns=["label"])
y = dataset["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

logging.info(f"Train size: {X_train.shape}")
logging.info(f"Test size: {X_test.shape}")

X_train.to_csv(f"{SAVE_DIR}/X_train_raw.csv", index=False)
X_test.to_csv(f"{SAVE_DIR}/X_test_raw.csv", index=False)
y_train.to_csv(f"{SAVE_DIR}/y_train.csv", index=False)
y_test.to_csv(f"{SAVE_DIR}/y_test.csv", index=False)


le = LabelEncoder()

X_train = X_train.copy()
X_test = X_test.copy()

X_train["species_id"] = le.fit_transform(X_train["species"])
X_test["species_id"] = le.transform(X_test["species"])

X_train = X_train.drop(columns=["species"])
X_test = X_test.drop(columns=["species"])

feature_names = X_train.columns.tolist()

logging.info(f"Feature count: {len(feature_names)}")
logging.info(f"Species count: {len(le.classes_)}")


imputer = SimpleImputer(strategy="median")

X_train_processed = pd.DataFrame(
    imputer.fit_transform(X_train),
    columns=feature_names,
    index=X_train.index
)

X_test_processed = pd.DataFrame(
    imputer.transform(X_test),
    columns=feature_names,
    index=X_test.index
)

logging.info(f"NaNs train: {X_train_processed.isna().sum().sum()}")
logging.info(f"NaNs test: {X_test_processed.isna().sum().sum()}")

X_train_processed.to_csv(f"{SAVE_DIR}/X_train_processed.csv", index=False)
X_test_processed.to_csv(f"{SAVE_DIR}/X_test_processed.csv", index=False)


model = get_model()

logging.info("Training enhanced LightGBM...")
model.fit(X_train_processed, y_train)

joblib.dump(model, f"{SAVE_DIR}/phyto_lightgbm_model.pkl")
joblib.dump(le, f"{SAVE_DIR}/species_encoder.pkl")
joblib.dump(imputer, f"{SAVE_DIR}/imputer.pkl")
joblib.dump(feature_names, f"{SAVE_DIR}/feature_names.pkl")

logging.info("Model + artifacts saved ✅")


y_pred = model.predict(X_test_processed)
y_prob = model.predict_proba(X_test_processed)[:, 1]

acc = accuracy_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_prob)
report = classification_report(y_test, y_pred)

logging.info(f"Accuracy: {acc:.4f}")
logging.info(f"AUC: {auc:.4f}")

print("\nClassification Report:\n")
print(report)

with open(f"{SAVE_DIR}/evaluation_report.txt", "w") as f:
    f.write(f"Accuracy: {acc:.4f}\n")
    f.write(f"AUC: {auc:.4f}\n\n")
    f.write(report)


importance_df = pd.DataFrame({
    "feature": feature_names,
    "importance": model.feature_importances_
}).sort_values("importance", ascending=False)

importance_df.to_csv(f"{SAVE_DIR}/feature_importance.csv", index=False)

print("\nTop 15 Features:")
print(importance_df.head(15))


def suitability_class(prob):
    if prob >= 0.75:
        return "Highly suitable"
    elif prob >= 0.50:
        return "Suitable"
    elif prob >= 0.30:
        return "Moderately suitable"
    else:
        return "Low / not suitable"


def predict_species_suitability(city_name, lat, lon, species_name):
    if species_name not in le.classes_:
        raise ValueError(f"{species_name} not found in training species list.")

    env_values = extractor.extract_point(lat, lon)

    base_cols = [f"BIO{i}" for i in range(1, 20)] + ["elevation"]

    sample_df = pd.DataFrame(
        [env_values],
        columns=base_cols
    )

    sample_df = add_engineered_features(sample_df)

    sample_df["species_id"] = le.transform([species_name])[0]
    sample_df = sample_df[feature_names]

    sample_processed = pd.DataFrame(
        imputer.transform(sample_df),
        columns=feature_names
    )

    prob = model.predict_proba(sample_processed)[0][1]
    label = int(prob >= 0.5)

    print("====================================")
    print(f"City: {city_name}")
    print(f"Species: {species_name}")
    print(f"Suitability Probability: {prob:.4f}")
    print(f"Suitability Class: {suitability_class(prob)}")
    print(f"Binary Prediction: {'Suitable' if label == 1 else 'Not suitable'}")
    print("====================================")

    return prob, label


predict_species_suitability(
    city_name="Cedars of God",
    lat=34.2436,
    lon=36.0486,
    species_name="Cedrus libani"
)

logging.info("Enhanced v4 pipeline completed successfully 🚀")

2026-05-10 06:01:21,535 - INFO - Logging is working ✅
2026-05-10 06:01:21,541 - INFO - Saving outputs to: /content/drive/MyDrive/PhytoLB/model_enhanced_v4
2026-05-10 06:01:21,615 - INFO - GBIF loaded: 13748 records
2026-05-10 06:01:21,617 - INFO - Unique species: 27
2026-05-10 06:01:21,746 - INFO - Climate + elevation rasters loaded ✅
2026-05-10 06:01:21,749 - INFO - Extracting presence features...
2026-05-10 06:01:21,767 - INFO - Processed 0/13748 points
2026-05-10 06:01:31,946 - INFO - Processed 1000/13748 points
2026-05-10 06:01:41,376 - INFO - Processed 2000/13748 points
2026-05-10 06:01:46,310 - INFO - Processed 3000/13748 points
2026-05-10 06:01:48,961 - INFO - Processed 4000/13748 points
2026-05-10 06:01:51,670 - INFO - Processed 5000/13748 points
2026-05-10 06:01:55,161 - INFO - Processed 6000/13748 points
2026-05-10 06:01:58,488 - INFO - Processed 7000/13748 points
2026-05-10 06:02:01,156 - INFO - Processed 8000/13748 points
2026-05-10 06:02:03,809 - INFO - Processed 9000/1374

In [ ]:
predict_species_suitability(
    city_name="Beirut",
    lat=33.8938,
    lon=35.5018,
    species_name="Cedrus libani"
)

City: Beirut
Species: Cedrus libani
Suitability Probability: 0.0342
Suitability Class: Low / not suitable
Binary Prediction: Not suitable


(np.float64(0.03419956541314632), 0)

In [ ]:
predict_species_suitability(
    city_name="Faraya",
    lat=33.9944,
    lon=35.8172,
    species_name="Abies cilicica"
)

City: Faraya
Species: Abies cilicica
Suitability Probability: 0.0158
Suitability Class: Low / not suitable
Binary Prediction: Not suitable


(np.float64(0.01581292199606067), 0)

In [ ]:
predict_species_suitability(
    city_name="Tyre",
    lat=33.2700,
    lon=35.2038,
    species_name="Abies cilicica"
)

City: Tyre
Species: Abies cilicica
Suitability Probability: 0.0001
Suitability Class: Low / not suitable
Binary Prediction: Not suitable


(np.float64(9.176412586832604e-05), 0)

In [ ]:
predict_species_suitability(
    city_name="Byblos",
    lat=34.1230,
    lon=35.6519,
    species_name="Quercus coccifera calliprinos"
)

City: Byblos
Species: Quercus coccifera calliprinos
Suitability Probability: 0.8501
Suitability Class: Highly suitable
Binary Prediction: Suitable


(np.float64(0.8501234326481469), 1)

In [ ]:
[x for x in le.classes_ if "Quercus" in x]

['Quercus cedrorum',
 'Quercus cerris',
 'Quercus coccifera calliprinos',
 'Quercus infectoria boissieri',
 'Quercus ithaburensis',
 'Quercus kotschyana',
 'Quercus look']

In [ ]:
predict_species_suitability(
    city_name="Zahle",
    lat=33.8463,
    lon=35.9020,
    species_name="Quercus ithaburensis"
)

City: Zahle
Species: Quercus ithaburensis
Suitability Probability: 0.0022
Suitability Class: Low / not suitable
Binary Prediction: Not suitable


(np.float64(0.0021704114998436596), 0)

In [ ]:
predict_species_suitability(
    city_name="Bcharre",
    lat=34.2508,
    lon=36.0106,
    species_name="Quercus infectoria boissieri"
)

City: Bcharre
Species: Quercus infectoria boissieri
Suitability Probability: 0.8926
Suitability Class: Highly suitable
Binary Prediction: Suitable


(np.float64(0.8926496313467402), 1)

In [ ]:
predict_species_suitability(
    city_name="Sidon",
    lat=33.5630,
    lon=35.3688,
    species_name="Ceratonia siliqua"
)

City: Sidon
Species: Ceratonia siliqua
Suitability Probability: 0.2775
Suitability Class: Low / not suitable
Binary Prediction: Not suitable


(np.float64(0.27751182647930783), 0)

In [ ]:
predict_species_suitability(
    city_name="Faraya",
    lat=33.9944,
    lon=35.8172,
    species_name="Ceratonia siliqua"
)

City: Faraya
Species: Ceratonia siliqua
Suitability Probability: 0.0143
Suitability Class: Low / not suitable
Binary Prediction: Not suitable


(np.float64(0.01428339155307848), 0)

In [ ]:
predict_species_suitability(
    city_name="Jbeil",
    lat=34.1217,
    lon=35.6481,
    species_name="Arbutus andrachne"
)

City: Jbeil
Species: Arbutus andrachne
Suitability Probability: 0.6640
Suitability Class: Suitable
Binary Prediction: Suitable


(np.float64(0.6639694479836706), 1)

In [ ]:
predict_species_suitability(
    city_name="Baalbek",
    lat=34.0060,
    lon=36.2181,
    species_name="Juniperus excelsa"
)

City: Baalbek
Species: Juniperus excelsa
Suitability Probability: 0.1370
Suitability Class: Low / not suitable
Binary Prediction: Not suitable


(np.float64(0.13700073602680754), 0)

In [ ]:
predict_species_suitability(
    city_name="Faraya",
    lat=33.9944,
    lon=35.8172,
    species_name="Juniperus drupacea"
)

City: Faraya
Species: Juniperus drupacea
Suitability Probability: 0.0779
Suitability Class: Low / not suitable
Binary Prediction: Not suitable


(np.float64(0.07792286859349763), 0)

In [ ]:
predict_species_suitability(
    city_name="Tyre",
    lat=33.2700,
    lon=35.2038,
    species_name="Cedrus libani"
)

City: Tyre
Species: Cedrus libani
Suitability Probability: 0.0006
Suitability Class: Low / not suitable
Binary Prediction: Not suitable


(np.float64(0.0005952741584186892), 0)

In [ ]:
predict_species_suitability(
    city_name="Bcharre",
    lat=34.2508,
    lon=36.0106,
    species_name="Ceratonia siliqua"
)

City: Bcharre
Species: Ceratonia siliqua
Suitability Probability: 0.1003
Suitability Class: Low / not suitable
Binary Prediction: Not suitable


(np.float64(0.1003403730393415), 0)

In [ ]:
predict_species_suitability(
    city_name="Beirut",
    lat=33.8938,
    lon=35.5018,
    species_name="Abies cilicica"
)

City: Beirut
Species: Abies cilicica
Suitability Probability: 0.0009
Suitability Class: Low / not suitable
Binary Prediction: Not suitable


(np.float64(0.0009157715537510458), 0)

In [ ]:
cities = [
    ("Beirut", 33.8938, 35.5018),
    ("Faraya", 33.9944, 35.8172),
    ("Bcharre", 34.2508, 36.0106),
    ("Tyre", 33.2700, 35.2038),
]

species_list = [
    "Cedrus libani",
    "Ceratonia siliqua",
    "Abies cilicica",
]

for city_name, lat, lon in cities:
    for sp in species_list:
        predict_species_suitability(
            city_name,
            lat,
            lon,
            sp
        )

City: Beirut
Species: Cedrus libani
Suitability Probability: 0.0342
Suitability Class: Low / not suitable
Binary Prediction: Not suitable
City: Beirut
Species: Ceratonia siliqua
Suitability Probability: 0.8009
Suitability Class: Highly suitable
Binary Prediction: Suitable
City: Beirut
Species: Abies cilicica
Suitability Probability: 0.0009
Suitability Class: Low / not suitable
Binary Prediction: Not suitable
City: Faraya
Species: Cedrus libani
Suitability Probability: 0.3607
Suitability Class: Moderately suitable
Binary Prediction: Not suitable
City: Faraya
Species: Ceratonia siliqua
Suitability Probability: 0.0143
Suitability Class: Low / not suitable
Binary Prediction: Not suitable
City: Faraya
Species: Abies cilicica
Suitability Probability: 0.0158
Suitability Class: Low / not suitable
Binary Prediction: Not suitable
City: Bcharre
Species: Cedrus libani
Suitability Probability: 0.5662
Suitability Class: Suitable
Binary Prediction: Suitable
City: Bcharre
Species: Ceratonia siliqua
Su

In [ ]:
abies = gbif[gbif["species"] == "Abies cilicica"]

print(abies.shape)

print(abies[["lat", "lon"]].describe())

(576, 3)
              lat         lon
count  576.000000  576.000000
mean    34.451117   36.191212
std      0.048232    0.059169
min     34.300800   35.985596
25%     34.412894   36.160012
50%     34.473156   36.216878
75%     34.490025   36.231046
max     34.517700   36.278457


## Enhanced V5

In [ ]:
# ============================================================
# PhytoLB Enhanced Pipeline v5
# Climate + Elevation + Engineered Features + LightGBM
# Saves datasets, model, artifacts, diagnostics, predictions
# ============================================================

# -----------------------
# 0. Imports
# -----------------------
import os
import sys
import joblib
import logging
import numpy as np
import pandas as pd
import rasterio
import lightgbm as lgb

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report, confusion_matrix


# -----------------------
# 1. Logging
# -----------------------
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    handlers=[logging.StreamHandler(sys.stdout)],
    force=True
)

logging.info("Logging is working ✅")


# -----------------------
# 2. Paths
# -----------------------
BASE_DIR = "/content/drive/MyDrive/PhytoLB"

GBIF_PATH = f"{BASE_DIR}/dwca-tree_lebanon-v1.0/occurrence.txt"
CLIMATE_DIR = f"{BASE_DIR}/wc2.1_10m_bio"
ELEVATION_PATH = f"{BASE_DIR}/elevation/elevation.tif"

SAVE_DIR = f"{BASE_DIR}/model_enhanced_v5"
os.makedirs(SAVE_DIR, exist_ok=True)

logging.info(f"Saving outputs to: {SAVE_DIR}")


# -----------------------
# 3. Load GBIF
# -----------------------
def load_gbif_txt(path):
    df = pd.read_csv(path, sep="\t", low_memory=False)

    if "species" not in df.columns:
        df["species"] = df["scientificName"]

    df = df[["species", "decimalLatitude", "decimalLongitude"]]
    df = df.dropna()

    df.columns = ["species", "lat", "lon"]

    df["lat"] = pd.to_numeric(df["lat"], errors="coerce")
    df["lon"] = pd.to_numeric(df["lon"], errors="coerce")
    df = df.dropna()

    # Lebanon approximate bounding box
    df = df[
        (df["lat"] >= 33.0) & (df["lat"] <= 34.8) &
        (df["lon"] >= 35.0) & (df["lon"] <= 36.8)
    ]

    df = df.drop_duplicates()

    logging.info(f"GBIF loaded: {len(df)} records")
    logging.info(f"Unique species: {df['species'].nunique()}")

    return df


# -----------------------
# 4. Environmental Extractor
# -----------------------
class EnvironmentalExtractor:
    def __init__(self, climate_folder, elevation_path):
        self.climate_rasters = []

        for i in range(1, 20):
            path = os.path.join(climate_folder, f"wc2.1_10m_bio_{i}.tif")

            if not os.path.exists(path):
                raise FileNotFoundError(f"Missing climate file: {path}")

            self.climate_rasters.append(rasterio.open(path))

        if not os.path.exists(elevation_path):
            raise FileNotFoundError(f"Missing elevation file: {elevation_path}")

        self.elevation_raster = rasterio.open(elevation_path)

        logging.info("Climate + elevation rasters loaded ✅")

    def extract_point(self, lat, lon):
        values = []

        for raster in self.climate_rasters:
            try:
                val = next(raster.sample([(lon, lat)]))[0]

                if raster.nodata is not None and val == raster.nodata:
                    val = np.nan

            except Exception:
                val = np.nan

            values.append(val)

        try:
            elev = next(self.elevation_raster.sample([(lon, lat)]))[0]

            if self.elevation_raster.nodata is not None and elev == self.elevation_raster.nodata:
                elev = np.nan

        except Exception:
            elev = np.nan

        values.append(elev)

        return values

    def extract_dataframe(self, df):
        features = []
        total = len(df)

        for i, row in enumerate(df.itertuples(index=False)):
            values = self.extract_point(row.lat, row.lon)
            features.append(values)

            if i % 1000 == 0:
                logging.info(f"Processed {i}/{total} points")

        columns = [f"BIO{i}" for i in range(1, 20)] + ["elevation"]

        return pd.DataFrame(features, columns=columns)


# -----------------------
# 5. Feature Engineering
# -----------------------
def add_engineered_features(df):
    df = df.copy()

    df["temp_annual_range"] = df["BIO5"] - df["BIO6"]
    df["warm_cold_quarter_diff"] = df["BIO10"] - df["BIO11"]

    df["precip_wet_dry_month_diff"] = df["BIO13"] - df["BIO14"]
    df["precip_wet_dry_quarter_diff"] = df["BIO16"] - df["BIO17"]

    df["dryness_ratio"] = df["BIO17"] / (df["BIO16"] + 1e-6)
    df["summer_winter_precip_ratio"] = df["BIO18"] / (df["BIO19"] + 1e-6)

    df["elevation_temp_interaction"] = df["elevation"] * df["BIO1"]
    df["elevation_precip_interaction"] = df["elevation"] * df["BIO12"]

    return df


# -----------------------
# 6. Filtered Pseudo-Absence Generation
# -----------------------
def generate_pseudo_absence(df, n_samples, extractor, random_state=42):
    np.random.seed(random_state)

    lat_min, lat_max = df["lat"].min(), df["lat"].max()
    lon_min, lon_max = df["lon"].min(), df["lon"].max()

    pseudo_rows = []
    attempts = 0
    max_attempts = n_samples * 20

    logging.info("Generating filtered pseudo-absence points...")

    while len(pseudo_rows) < n_samples and attempts < max_attempts:
        attempts += 1

        lat = np.random.uniform(lat_min, lat_max)
        lon = np.random.uniform(lon_min, lon_max)

        values = extractor.extract_point(lat, lon)

        # Reject invalid environmental points
        if np.isnan(values).sum() > 0:
            continue

        elevation = values[-1]

        # Reject sea / invalid terrain
        if elevation < 0 or elevation > 3200:
            continue

        species = df["species"].sample(
            1,
            random_state=np.random.randint(0, 999999)
        ).iloc[0]

        pseudo_rows.append({
            "species": species,
            "lat": lat,
            "lon": lon,
            "label": 0
        })

        if len(pseudo_rows) % 1000 == 0:
            logging.info(f"Generated {len(pseudo_rows)}/{n_samples} valid pseudo-absences")

    pseudo_df = pd.DataFrame(pseudo_rows)

    if len(pseudo_df) < n_samples:
        logging.warning(f"Only generated {len(pseudo_df)} pseudo-absences out of {n_samples}")

    return pseudo_df


# -----------------------
# 7. Build Dataset
# -----------------------
def build_dataset(df, extractor):
    logging.info("Extracting presence features...")

    presence_features = extractor.extract_dataframe(df)
    presence_features["species"] = df["species"].values
    presence_features["lat"] = df["lat"].values
    presence_features["lon"] = df["lon"].values
    presence_features["label"] = 1

    pseudo_df = generate_pseudo_absence(
        df=df,
        n_samples=len(df),
        extractor=extractor,
        random_state=42
    )

    logging.info("Extracting pseudo-absence features...")

    absence_features = extractor.extract_dataframe(pseudo_df)
    absence_features["species"] = pseudo_df["species"].values
    absence_features["lat"] = pseudo_df["lat"].values
    absence_features["lon"] = pseudo_df["lon"].values
    absence_features["label"] = 0

    full_df = pd.concat([presence_features, absence_features], ignore_index=True)

    logging.info(f"Dataset before filtering: {full_df.shape}")

    env_cols = [f"BIO{i}" for i in range(1, 20)] + ["elevation"]
    full_df = full_df.dropna(subset=env_cols)

    logging.info(f"Dataset after filtering: {full_df.shape}")

    full_df = add_engineered_features(full_df)

    return full_df


# -----------------------
# 8. Model
# -----------------------
def get_model():
    return lgb.LGBMClassifier(
        objective="binary",
        n_estimators=900,
        learning_rate=0.02,
        max_depth=9,
        num_leaves=80,
        min_child_samples=15,
        subsample=0.9,
        subsample_freq=1,
        colsample_bytree=0.9,
        reg_alpha=0.05,
        reg_lambda=0.3,
        random_state=42,
        force_row_wise=True,
        verbose=-1
    )


# -----------------------
# 9. Helpers
# -----------------------
def suitability_class(prob):
    if prob >= 0.75:
        return "Highly suitable"
    elif prob >= 0.50:
        return "Suitable"
    elif prob >= 0.30:
        return "Moderately suitable"
    else:
        return "Low / not suitable"


def find_species(name_part):
    matches = [sp for sp in le.classes_ if name_part.lower() in sp.lower()]

    if not matches:
        print("No species found.")
    else:
        for sp in matches:
            print(sp)


def species_distribution(species_name):
    subset = gbif[gbif["species"] == species_name]

    if subset.empty:
        print(f"{species_name} not found.")
        return None

    print(f"\nSpecies: {species_name}")
    print(f"Records: {len(subset)}")
    print(subset[["lat", "lon"]].describe())

    return subset


# ============================================================
# MAIN PIPELINE
# ============================================================

# -----------------------
# 10. Load Data
# -----------------------
gbif = load_gbif_txt(GBIF_PATH)
gbif.to_csv(f"{SAVE_DIR}/clean_gbif.csv", index=False)

species_counts = gbif["species"].value_counts().reset_index()
species_counts.columns = ["species", "records"]
species_counts.to_csv(f"{SAVE_DIR}/species_counts.csv", index=False)


# -----------------------
# 11. Initialize Extractor
# -----------------------
extractor = EnvironmentalExtractor(
    climate_folder=CLIMATE_DIR,
    elevation_path=ELEVATION_PATH
)


# -----------------------
# 12. Build Dataset
# -----------------------
dataset = build_dataset(gbif, extractor)

logging.info(f"Final dataset shape: {dataset.shape}")
logging.info(f"Class distribution:\n{dataset['label'].value_counts()}")
logging.info(f"Missing values total: {dataset.isna().sum().sum()}")

dataset.to_csv(f"{SAVE_DIR}/final_phyto_dataset.csv", index=False)
dataset.to_parquet(f"{SAVE_DIR}/final_phyto_dataset.parquet", index=False)


# -----------------------
# 13. Split
# -----------------------
X = dataset.drop(columns=["label"])
y = dataset["label"]

# Keep coordinates for diagnostics, but remove from model features later
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

logging.info(f"Train size: {X_train.shape}")
logging.info(f"Test size: {X_test.shape}")

X_train.to_csv(f"{SAVE_DIR}/X_train_raw.csv", index=False)
X_test.to_csv(f"{SAVE_DIR}/X_test_raw.csv", index=False)
y_train.to_csv(f"{SAVE_DIR}/y_train.csv", index=False)
y_test.to_csv(f"{SAVE_DIR}/y_test.csv", index=False)


# -----------------------
# 14. Encode Species
# -----------------------
le = LabelEncoder()

X_train = X_train.copy()
X_test = X_test.copy()

X_train["species_id"] = le.fit_transform(X_train["species"])
X_test["species_id"] = le.transform(X_test["species"])

# Save diagnostic columns separately
train_diagnostics = X_train[["species", "lat", "lon"]].copy()
test_diagnostics = X_test[["species", "lat", "lon"]].copy()

train_diagnostics.to_csv(f"{SAVE_DIR}/train_diagnostics.csv", index=False)
test_diagnostics.to_csv(f"{SAVE_DIR}/test_diagnostics.csv", index=False)

# Drop non-model columns
X_train = X_train.drop(columns=["species", "lat", "lon"])
X_test = X_test.drop(columns=["species", "lat", "lon"])

feature_names = X_train.columns.tolist()

logging.info(f"Feature count: {len(feature_names)}")
logging.info(f"Species count: {len(le.classes_)}")


# -----------------------
# 15. Impute
# -----------------------
imputer = SimpleImputer(strategy="median")

X_train_processed = pd.DataFrame(
    imputer.fit_transform(X_train),
    columns=feature_names,
    index=X_train.index
)

X_test_processed = pd.DataFrame(
    imputer.transform(X_test),
    columns=feature_names,
    index=X_test.index
)

logging.info(f"NaNs train: {X_train_processed.isna().sum().sum()}")
logging.info(f"NaNs test: {X_test_processed.isna().sum().sum()}")

X_train_processed.to_csv(f"{SAVE_DIR}/X_train_processed.csv", index=False)
X_test_processed.to_csv(f"{SAVE_DIR}/X_test_processed.csv", index=False)


# -----------------------
# 16. Train
# -----------------------
model = get_model()

logging.info("Training enhanced LightGBM...")
model.fit(X_train_processed, y_train)

joblib.dump(model, f"{SAVE_DIR}/phyto_lightgbm_model.pkl")
joblib.dump(le, f"{SAVE_DIR}/species_encoder.pkl")
joblib.dump(imputer, f"{SAVE_DIR}/imputer.pkl")
joblib.dump(feature_names, f"{SAVE_DIR}/feature_names.pkl")

logging.info("Model + artifacts saved ✅")


# -----------------------
# 17. Evaluate
# -----------------------
y_pred = model.predict(X_test_processed)
y_prob = model.predict_proba(X_test_processed)[:, 1]

acc = accuracy_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_prob)
report = classification_report(y_test, y_pred)
cm = confusion_matrix(y_test, y_pred)

logging.info(f"Accuracy: {acc:.4f}")
logging.info(f"AUC: {auc:.4f}")

print("\nClassification Report:\n")
print(report)

print("\nConfusion Matrix:")
print(cm)

with open(f"{SAVE_DIR}/evaluation_report.txt", "w") as f:
    f.write(f"Accuracy: {acc:.4f}\n")
    f.write(f"AUC: {auc:.4f}\n\n")
    f.write("Classification Report:\n")
    f.write(report)
    f.write("\nConfusion Matrix:\n")
    f.write(str(cm))


# -----------------------
# 18. Feature Importance
# -----------------------
importance_df = pd.DataFrame({
    "feature": feature_names,
    "importance": model.feature_importances_
}).sort_values("importance", ascending=False)

importance_df.to_csv(f"{SAVE_DIR}/feature_importance.csv", index=False)

print("\nTop 15 Features:")
print(importance_df.head(15))


# -----------------------
# 19. Prediction Function
# -----------------------
def predict_species_suitability(city_name, lat, lon, species_name):
    if species_name not in le.classes_:
        print(f"'{species_name}' not found.")
        print("Closest matches:")
        for sp in le.classes_:
            if species_name.split()[0].lower() in sp.lower():
                print("-", sp)
        return None

    env_values = extractor.extract_point(lat, lon)

    base_cols = [f"BIO{i}" for i in range(1, 20)] + ["elevation"]

    sample_df = pd.DataFrame([env_values], columns=base_cols)

    sample_df = add_engineered_features(sample_df)

    sample_df["species_id"] = le.transform([species_name])[0]
    sample_df = sample_df[feature_names]

    sample_processed = pd.DataFrame(
        imputer.transform(sample_df),
        columns=feature_names
    )

    prob = model.predict_proba(sample_processed)[0][1]
    label = int(prob >= 0.5)

    print("====================================")
    print(f"City: {city_name}")
    print(f"Species: {species_name}")
    print(f"Suitability Probability: {prob:.4f}")
    print(f"Suitability Class: {suitability_class(prob)}")
    print(f"Binary Prediction: {'Suitable' if label == 1 else 'Not suitable'}")
    print("====================================")

    return float(prob), label


# -----------------------
# 20. Batch Prediction Function
# -----------------------
def batch_predict(cities, species_list):
    results = []

    for city_name, lat, lon in cities:
        for species_name in species_list:
            output = predict_species_suitability(city_name, lat, lon, species_name)

            if output is None:
                continue

            prob, label = output

            results.append({
                "city": city_name,
                "lat": lat,
                "lon": lon,
                "species": species_name,
                "probability": prob,
                "class": suitability_class(prob),
                "binary_label": label
            })

    results_df = pd.DataFrame(results)
    results_df.to_csv(f"{SAVE_DIR}/city_species_predictions.csv", index=False)

    return results_df


# -----------------------
# 21. Test Samples
# -----------------------
test_cities = [
    ("Beirut", 33.8938, 35.5018),
    ("Tripoli", 34.4367, 35.8497),
    ("Byblos", 34.1230, 35.6519),
    ("Bcharre", 34.2508, 36.0106),
    ("Cedars of God", 34.2436, 36.0486),
    ("Faraya", 33.9944, 35.8172),
    ("Zahle", 33.8463, 35.9020),
    ("Baalbek", 34.0060, 36.2181),
    ("Sidon", 33.5630, 35.3688),
    ("Tyre", 33.2700, 35.2038)
]

test_species = [
    "Cedrus libani",
    "Ceratonia siliqua",
    "Abies cilicica",
    "Quercus coccifera calliprinos",
    "Quercus infectoria boissieri",
    "Arbutus andrachne",
    "Juniperus excelsa",
    "Juniperus drupacea"
]

predictions_df = batch_predict(test_cities, test_species)

print("\nPrediction summary:")
print(predictions_df.sort_values("probability", ascending=False).head(20))

logging.info("Enhanced v5 pipeline completed successfully 🚀")

2026-05-10 06:21:57,612 - INFO - Logging is working ✅
2026-05-10 06:21:57,630 - INFO - Saving outputs to: /content/drive/MyDrive/PhytoLB/model_enhanced_v5
2026-05-10 06:21:57,744 - INFO - GBIF loaded: 13748 records
2026-05-10 06:21:57,747 - INFO - Unique species: 27
2026-05-10 06:21:57,928 - INFO - Climate + elevation rasters loaded ✅
2026-05-10 06:21:57,935 - INFO - Extracting presence features...
2026-05-10 06:21:57,955 - INFO - Processed 0/13748 points
2026-05-10 06:22:00,958 - INFO - Processed 1000/13748 points
2026-05-10 06:22:03,617 - INFO - Processed 2000/13748 points
2026-05-10 06:22:06,366 - INFO - Processed 3000/13748 points
2026-05-10 06:22:09,505 - INFO - Processed 4000/13748 points
2026-05-10 06:22:13,611 - INFO - Processed 5000/13748 points
2026-05-10 06:22:16,323 - INFO - Processed 6000/13748 points
2026-05-10 06:22:19,131 - INFO - Processed 7000/13748 points
2026-05-10 06:22:21,974 - INFO - Processed 8000/13748 points
2026-05-10 06:22:25,525 - INFO - Processed 9000/1374

# Catboost

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# ============================================================
# PhytoLB CatBoost Pipeline v1
# Climate + Elevation + Engineered Features + CatBoost
# ============================================================

# -----------------------
# 0. Install CatBoost
# -----------------------
!pip install catboost -q

# -----------------------
# 1. Imports
# -----------------------
import os
import sys
import joblib
import logging
import numpy as np
import pandas as pd
import rasterio

from catboost import CatBoostClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    classification_report,
    confusion_matrix
)

# -----------------------
# 2. Logging
# -----------------------
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    handlers=[logging.StreamHandler(sys.stdout)],
    force=True
)

logging.info("Logging is working ✅")

# -----------------------
# 3. Paths
# -----------------------
BASE_DIR = "/content/drive/MyDrive/PhytoLB"

GBIF_PATH = f"{BASE_DIR}/dwca-tree_lebanon-v1.0/occurrence.txt"
CLIMATE_DIR = f"{BASE_DIR}/wc2.1_10m_bio"
ELEVATION_PATH = f"{BASE_DIR}/elevation/elevation.tif"

SAVE_DIR = f"{BASE_DIR}/model_catboost_v1"
os.makedirs(SAVE_DIR, exist_ok=True)

logging.info(f"Saving outputs to: {SAVE_DIR}")


# -----------------------
# 4. Load GBIF
# -----------------------
def load_gbif_txt(path):
    df = pd.read_csv(path, sep="\t", low_memory=False)

    if "species" not in df.columns:
        df["species"] = df["scientificName"]

    df = df[["species", "decimalLatitude", "decimalLongitude"]]
    df = df.dropna()

    df.columns = ["species", "lat", "lon"]

    df["lat"] = pd.to_numeric(df["lat"], errors="coerce")
    df["lon"] = pd.to_numeric(df["lon"], errors="coerce")
    df = df.dropna()

    df = df[
        (df["lat"] >= 33.0) & (df["lat"] <= 34.8) &
        (df["lon"] >= 35.0) & (df["lon"] <= 36.8)
    ]

    df = df.drop_duplicates()

    logging.info(f"GBIF loaded: {len(df)} records")
    logging.info(f"Unique species: {df['species'].nunique()}")

    return df


# -----------------------
# 5. Environmental Extractor
# -----------------------
class EnvironmentalExtractor:
    def __init__(self, climate_folder, elevation_path):
        self.climate_rasters = []

        for i in range(1, 20):
            path = os.path.join(climate_folder, f"wc2.1_10m_bio_{i}.tif")

            if not os.path.exists(path):
                raise FileNotFoundError(f"Missing climate file: {path}")

            self.climate_rasters.append(rasterio.open(path))

        if not os.path.exists(elevation_path):
            raise FileNotFoundError(f"Missing elevation file: {elevation_path}")

        self.elevation_raster = rasterio.open(elevation_path)

        logging.info("Climate + elevation rasters loaded ✅")

    def extract_point(self, lat, lon):
        values = []

        for raster in self.climate_rasters:
            try:
                val = next(raster.sample([(lon, lat)]))[0]

                if raster.nodata is not None and val == raster.nodata:
                    val = np.nan

            except Exception:
                val = np.nan

            values.append(val)

        try:
            elev = next(self.elevation_raster.sample([(lon, lat)]))[0]

            if self.elevation_raster.nodata is not None and elev == self.elevation_raster.nodata:
                elev = np.nan

        except Exception:
            elev = np.nan

        values.append(elev)

        return values

    def extract_dataframe(self, df):
        features = []
        total = len(df)

        for i, row in enumerate(df.itertuples(index=False)):
            values = self.extract_point(row.lat, row.lon)
            features.append(values)

            if i % 1000 == 0:
                logging.info(f"Processed {i}/{total} points")

        columns = [f"BIO{i}" for i in range(1, 20)] + ["elevation"]

        return pd.DataFrame(features, columns=columns)


# -----------------------
# 6. Feature Engineering
# -----------------------
def add_engineered_features(df):
    df = df.copy()

    df["temp_annual_range"] = df["BIO5"] - df["BIO6"]
    df["warm_cold_quarter_diff"] = df["BIO10"] - df["BIO11"]

    df["precip_wet_dry_month_diff"] = df["BIO13"] - df["BIO14"]
    df["precip_wet_dry_quarter_diff"] = df["BIO16"] - df["BIO17"]

    df["dryness_ratio"] = df["BIO17"] / (df["BIO16"] + 1e-6)
    df["summer_winter_precip_ratio"] = df["BIO18"] / (df["BIO19"] + 1e-6)

    df["elevation_temp_interaction"] = df["elevation"] * df["BIO1"]
    df["elevation_precip_interaction"] = df["elevation"] * df["BIO12"]

    return df


# -----------------------
# 7. Filtered Pseudo-Absence
# -----------------------
def generate_pseudo_absence(df, n_samples, extractor, random_state=42):
    np.random.seed(random_state)

    lat_min, lat_max = df["lat"].min(), df["lat"].max()
    lon_min, lon_max = df["lon"].min(), df["lon"].max()

    pseudo_rows = []
    attempts = 0
    max_attempts = n_samples * 20

    logging.info("Generating filtered pseudo-absence points...")

    while len(pseudo_rows) < n_samples and attempts < max_attempts:
        attempts += 1

        lat = np.random.uniform(lat_min, lat_max)
        lon = np.random.uniform(lon_min, lon_max)

        values = extractor.extract_point(lat, lon)

        if np.isnan(values).sum() > 0:
            continue

        elevation = values[-1]

        if elevation < 0 or elevation > 3200:
            continue

        species = df["species"].sample(
            1,
            random_state=np.random.randint(0, 999999)
        ).iloc[0]

        pseudo_rows.append({
            "species": species,
            "lat": lat,
            "lon": lon,
            "label": 0
        })

        if len(pseudo_rows) % 1000 == 0:
            logging.info(f"Generated {len(pseudo_rows)}/{n_samples} valid pseudo-absences")

    pseudo_df = pd.DataFrame(pseudo_rows)

    if len(pseudo_df) < n_samples:
        logging.warning(f"Only generated {len(pseudo_df)} pseudo-absences out of {n_samples}")

    return pseudo_df


# -----------------------
# 8. Build Dataset
# -----------------------
def build_dataset(df, extractor):
    logging.info("Extracting presence features...")

    presence_features = extractor.extract_dataframe(df)
    presence_features["species"] = df["species"].values
    presence_features["lat"] = df["lat"].values
    presence_features["lon"] = df["lon"].values
    presence_features["label"] = 1

    pseudo_df = generate_pseudo_absence(
        df=df,
        n_samples=len(df),
        extractor=extractor,
        random_state=42
    )

    logging.info("Extracting pseudo-absence features...")

    absence_features = extractor.extract_dataframe(pseudo_df)
    absence_features["species"] = pseudo_df["species"].values
    absence_features["lat"] = pseudo_df["lat"].values
    absence_features["lon"] = pseudo_df["lon"].values
    absence_features["label"] = 0

    full_df = pd.concat([presence_features, absence_features], ignore_index=True)

    logging.info(f"Dataset before filtering: {full_df.shape}")

    env_cols = [f"BIO{i}" for i in range(1, 20)] + ["elevation"]
    full_df = full_df.dropna(subset=env_cols)

    full_df = add_engineered_features(full_df)

    logging.info(f"Dataset after feature engineering: {full_df.shape}")

    return full_df


# -----------------------
# 9. Helper Functions
# -----------------------
def suitability_class(prob):
    if prob >= 0.75:
        return "Highly suitable"
    elif prob >= 0.50:
        return "Suitable"
    elif prob >= 0.30:
        return "Moderately suitable"
    else:
        return "Low / not suitable"


def find_species(name_part):
    matches = [sp for sp in le.classes_ if name_part.lower() in sp.lower()]

    if not matches:
        print("No species found.")
    else:
        for sp in matches:
            print(sp)


# ============================================================
# MAIN PIPELINE
# ============================================================

# -----------------------
# 10. Load Data
# -----------------------
gbif = load_gbif_txt(GBIF_PATH)
gbif.to_csv(f"{SAVE_DIR}/clean_gbif.csv", index=False)

species_counts = gbif["species"].value_counts().reset_index()
species_counts.columns = ["species", "records"]
species_counts.to_csv(f"{SAVE_DIR}/species_counts.csv", index=False)


# -----------------------
# 11. Init Extractor
# -----------------------
extractor = EnvironmentalExtractor(
    climate_folder=CLIMATE_DIR,
    elevation_path=ELEVATION_PATH
)


# -----------------------
# 12. Build Dataset
# -----------------------
dataset = build_dataset(gbif, extractor)

logging.info(f"Final dataset shape: {dataset.shape}")
logging.info(f"Class distribution:\n{dataset['label'].value_counts()}")
logging.info(f"Missing values total: {dataset.isna().sum().sum()}")

dataset.to_csv(f"{SAVE_DIR}/catboost_phyto_dataset.csv", index=False)
dataset.to_parquet(f"{SAVE_DIR}/catboost_phyto_dataset.parquet", index=False)


# -----------------------
# 13. Split
# -----------------------
X = dataset.drop(columns=["label"])
y = dataset["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

logging.info(f"Train size: {X_train.shape}")
logging.info(f"Test size: {X_test.shape}")


# -----------------------
# 14. Prepare Features for CatBoost
# -----------------------
X_train = X_train.copy()
X_test = X_test.copy()

# Keep diagnostics
X_train[["species", "lat", "lon"]].to_csv(f"{SAVE_DIR}/train_diagnostics.csv", index=False)
X_test[["species", "lat", "lon"]].to_csv(f"{SAVE_DIR}/test_diagnostics.csv", index=False)

# Remove coordinates to avoid direct geographic memorization
X_train = X_train.drop(columns=["lat", "lon"])
X_test = X_test.drop(columns=["lat", "lon"])

# For CatBoost, keep species as categorical string
cat_features = ["species"]

feature_names = X_train.columns.tolist()

logging.info(f"Feature count: {len(feature_names)}")
logging.info(f"Categorical features: {cat_features}")


# -----------------------
# 15. Impute Numeric Features
# -----------------------
numeric_features = [col for col in feature_names if col not in cat_features]

imputer = SimpleImputer(strategy="median")

X_train[numeric_features] = imputer.fit_transform(X_train[numeric_features])
X_test[numeric_features] = imputer.transform(X_test[numeric_features])

logging.info(f"NaNs train: {X_train.isna().sum().sum()}")
logging.info(f"NaNs test: {X_test.isna().sum().sum()}")

X_train.to_csv(f"{SAVE_DIR}/X_train_catboost_processed.csv", index=False)
X_test.to_csv(f"{SAVE_DIR}/X_test_catboost_processed.csv", index=False)
y_train.to_csv(f"{SAVE_DIR}/y_train.csv", index=False)
y_test.to_csv(f"{SAVE_DIR}/y_test.csv", index=False)


# -----------------------
# 16. Train CatBoost
# -----------------------
cat_model = CatBoostClassifier(
    iterations=1500,
    learning_rate=0.025,
    depth=8,
    loss_function="Logloss",
    eval_metric="AUC",
    random_seed=42,
    l2_leaf_reg=3,
    subsample=0.9,
    bootstrap_type="Bernoulli",
    verbose=100
)

logging.info("Training CatBoost model...")

cat_model.fit(
    X_train,
    y_train,
    cat_features=cat_features,
    eval_set=(X_test, y_test),
    use_best_model=True
)

logging.info("CatBoost training completed ✅")


# -----------------------
# 17. Save Model + Artifacts
# -----------------------
cat_model.save_model(f"{SAVE_DIR}/catboost_phyto_model.cbm")
joblib.dump(cat_model, f"{SAVE_DIR}/catboost_phyto_model.pkl")
joblib.dump(imputer, f"{SAVE_DIR}/numeric_imputer.pkl")
joblib.dump(feature_names, f"{SAVE_DIR}/feature_names.pkl")
joblib.dump(cat_features, f"{SAVE_DIR}/cat_features.pkl")
joblib.dump(numeric_features, f"{SAVE_DIR}/numeric_features.pkl")

logging.info("CatBoost model + artifacts saved ✅")


# -----------------------
# 18. Evaluate
# -----------------------
y_pred = cat_model.predict(X_test)
y_prob = cat_model.predict_proba(X_test)[:, 1]

acc = accuracy_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_prob)
report = classification_report(y_test, y_pred)
cm = confusion_matrix(y_test, y_pred)

logging.info(f"CatBoost Accuracy: {acc:.4f}")
logging.info(f"CatBoost AUC: {auc:.4f}")

print("\nClassification Report:\n")
print(report)

print("\nConfusion Matrix:")
print(cm)

with open(f"{SAVE_DIR}/catboost_evaluation_report.txt", "w") as f:
    f.write(f"CatBoost Accuracy: {acc:.4f}\n")
    f.write(f"CatBoost AUC: {auc:.4f}\n\n")
    f.write("Classification Report:\n")
    f.write(report)
    f.write("\nConfusion Matrix:\n")
    f.write(str(cm))


# -----------------------
# 19. Feature Importance
# -----------------------
importance_values = cat_model.get_feature_importance()

importance_df = pd.DataFrame({
    "feature": feature_names,
    "importance": importance_values
}).sort_values("importance", ascending=False)

importance_df.to_csv(f"{SAVE_DIR}/catboost_feature_importance.csv", index=False)

print("\nTop 15 CatBoost Features:")
print(importance_df.head(15))


# -----------------------
# 20. Prediction Function
# -----------------------
def predict_species_suitability_catboost(city_name, lat, lon, species_name):
    if species_name not in gbif["species"].unique():
        print(f"'{species_name}' not found.")
        print("Closest matches:")
        for sp in sorted(gbif["species"].unique()):
            if species_name.split()[0].lower() in sp.lower():
                print("-", sp)
        return None

    env_values = extractor.extract_point(lat, lon)

    base_cols = [f"BIO{i}" for i in range(1, 20)] + ["elevation"]

    sample_df = pd.DataFrame([env_values], columns=base_cols)
    sample_df = add_engineered_features(sample_df)

    sample_df["species"] = species_name

    # Match training columns
    sample_df = sample_df[feature_names]

    sample_df[numeric_features] = imputer.transform(sample_df[numeric_features])

    prob = cat_model.predict_proba(sample_df)[0][1]
    label = int(prob >= 0.5)

    print("====================================")
    print(f"City: {city_name}")
    print(f"Species: {species_name}")
    print(f"Suitability Probability: {prob:.4f}")
    print(f"Suitability Class: {suitability_class(prob)}")
    print(f"Binary Prediction: {'Suitable' if label == 1 else 'Not suitable'}")
    print("====================================")

    return float(prob), label


# -----------------------
# 21. Batch Prediction
# -----------------------
def batch_predict_catboost(cities, species_list):
    results = []

    for city_name, lat, lon in cities:
        for species_name in species_list:
            output = predict_species_suitability_catboost(
                city_name,
                lat,
                lon,
                species_name
            )

            if output is None:
                continue

            prob, label = output

            results.append({
                "city": city_name,
                "lat": lat,
                "lon": lon,
                "species": species_name,
                "probability": prob,
                "class": suitability_class(prob),
                "binary_label": label
            })

    results_df = pd.DataFrame(results)
    results_df.to_csv(f"{SAVE_DIR}/catboost_city_species_predictions.csv", index=False)

    return results_df


# -----------------------
# 22. Test Cities and Species
# -----------------------
test_cities = [
    ("Beirut", 33.8938, 35.5018),
    ("Tripoli", 34.4367, 35.8497),
    ("Byblos", 34.1230, 35.6519),
    ("Bcharre", 34.2508, 36.0106),
    ("Cedars of God", 34.2436, 36.0486),
    ("Faraya", 33.9944, 35.8172),
    ("Zahle", 33.8463, 35.9020),
    ("Baalbek", 34.0060, 36.2181),
    ("Sidon", 33.5630, 35.3688),
    ("Tyre", 33.2700, 35.2038)
]

test_species = [
    "Cedrus libani",
    "Ceratonia siliqua",
    "Abies cilicica",
    "Quercus coccifera calliprinos",
    "Quercus infectoria boissieri",
    "Arbutus andrachne",
    "Juniperus excelsa",
    "Juniperus drupacea"
]

predictions_df = batch_predict_catboost(test_cities, test_species)

print("\nPrediction summary:")
print(predictions_df.sort_values("probability", ascending=False).head(20))

logging.info("CatBoost pipeline completed successfully 🚀")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 8.3 MB/s eta 0:00:00
2026-05-10 06:40:48,214 - INFO - Logging is working ✅
2026-05-10 06:40:48,222 - INFO - Saving outputs to: /content/drive/MyDrive/PhytoLB/model_catboost_v1
2026-05-10 06:40:48,292 - INFO - GBIF loaded: 13748 records
2026-05-10 06:40:48,294 - INFO - Unique species: 27
2026-05-10 06:40:48,438 - INFO - Climate + elevation rasters loaded ✅
2026-05-10 06:40:48,443 - INFO - Extracting presence features...
2026-05-10 06:40:48,459 - INFO - Processed 0/13748 points
2026-05-10 06:40:51,245 - INFO - Processed 1000/13748 points
2026-05-10 06:40:55,226 - INFO - Processed 2000/13748 points
2026-05-10 06:40:58,130 - INFO - Processed 3000/13748 points
2026-05-10 06:41:00,975 - INFO - Processed 4000/13748 points
2026-05-10 06:41:03,756 - INFO - Processed 5000/13748 points
2026-05-10 06:41:06,920 - INFO - Processed 6000/13748 points
2026-05-10 06:41:10,823 - INFO - Processed 7000/13748 points
2026-05-10 06:41:13,483 - INFO - Pr

# Testing 4 models

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install catboost -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 9.1 MB/s eta 0:00:00


In [ ]:
# ============================================================
# Compare Saved PhytoLB Models
# ============================================================

import os
import joblib
import numpy as np
import pandas as pd
import rasterio

from catboost import CatBoostClassifier

# -----------------------
# 1. Paths
# -----------------------
BASE_DIR = "/content/drive/MyDrive/PhytoLB"

MODEL_DIRS = {
    "CatBoost_v1": f"{BASE_DIR}/model_catboost_v1",
    "LightGBM_v5": f"{BASE_DIR}/model_enhanced_v5_BETTER_IN_LAT_LONG_acc92_auc_96",
    "LightGBM_v4": f"{BASE_DIR}/model_enhanced_v4_BEST_IN_PREDICTION_acc92_auc_96",
    "LightGBM_v3": f"{BASE_DIR}/model_enhanced_v3_acc93_auc97",
}

CLIMATE_DIR = f"{BASE_DIR}/wc2.1_10m_bio"
ELEVATION_PATH = f"{BASE_DIR}/elevation/elevation.tif"


# -----------------------
# 2. Environmental Extractor
# -----------------------
class EnvironmentalExtractor:
    def __init__(self, climate_folder, elevation_path):
        self.climate_rasters = []

        for i in range(1, 20):
            path = os.path.join(climate_folder, f"wc2.1_10m_bio_{i}.tif")
            self.climate_rasters.append(rasterio.open(path))

        self.elevation_raster = rasterio.open(elevation_path)

    def extract_point(self, lat, lon):
        values = []

        for raster in self.climate_rasters:
            try:
                val = next(raster.sample([(lon, lat)]))[0]
                if raster.nodata is not None and val == raster.nodata:
                    val = np.nan
            except:
                val = np.nan

            values.append(val)

        try:
            elev = next(self.elevation_raster.sample([(lon, lat)]))[0]
            if self.elevation_raster.nodata is not None and elev == self.elevation_raster.nodata:
                elev = np.nan
        except:
            elev = np.nan

        values.append(elev)

        return values


extractor = EnvironmentalExtractor(CLIMATE_DIR, ELEVATION_PATH)


# -----------------------
# 3. Feature Engineering
# -----------------------
def add_engineered_features(df):
    df = df.copy()

    if "elevation" in df.columns:
        df["temp_annual_range"] = df["BIO5"] - df["BIO6"]
        df["warm_cold_quarter_diff"] = df["BIO10"] - df["BIO11"]

        df["precip_wet_dry_month_diff"] = df["BIO13"] - df["BIO14"]
        df["precip_wet_dry_quarter_diff"] = df["BIO16"] - df["BIO17"]

        df["dryness_ratio"] = df["BIO17"] / (df["BIO16"] + 1e-6)
        df["summer_winter_precip_ratio"] = df["BIO18"] / (df["BIO19"] + 1e-6)

        df["elevation_temp_interaction"] = df["elevation"] * df["BIO1"]
        df["elevation_precip_interaction"] = df["elevation"] * df["BIO12"]

    return df


def suitability_class(prob):
    if prob >= 0.75:
        return "Highly suitable"
    elif prob >= 0.50:
        return "Suitable"
    elif prob >= 0.30:
        return "Moderately suitable"
    else:
        return "Low / not suitable"


# -----------------------
# 4. Load Models
# -----------------------
def load_model_bundle(model_name, model_dir):
    bundle = {"name": model_name, "dir": model_dir}

    if model_name.startswith("CatBoost"):
        model_path = f"{model_dir}/catboost_phyto_model.cbm"

        model = CatBoostClassifier()
        model.load_model(model_path)

        bundle["type"] = "catboost"
        bundle["model"] = model
        bundle["imputer"] = joblib.load(f"{model_dir}/numeric_imputer.pkl")
        bundle["feature_names"] = joblib.load(f"{model_dir}/feature_names.pkl")
        bundle["numeric_features"] = joblib.load(f"{model_dir}/numeric_features.pkl")
        bundle["cat_features"] = joblib.load(f"{model_dir}/cat_features.pkl")

    else:
        bundle["type"] = "lightgbm"
        bundle["model"] = joblib.load(f"{model_dir}/phyto_lightgbm_model.pkl")
        bundle["encoder"] = joblib.load(f"{model_dir}/species_encoder.pkl")
        bundle["imputer"] = joblib.load(f"{model_dir}/imputer.pkl")
        bundle["feature_names"] = joblib.load(f"{model_dir}/feature_names.pkl")

    return bundle


models = []

for name, path in MODEL_DIRS.items():
    print("Loading:", name)
    models.append(load_model_bundle(name, path))

print("Loaded models:", [m["name"] for m in models])


# -----------------------
# 5. Predict With One Model
# -----------------------
def predict_with_model(bundle, city_name, lat, lon, species_name):
    env_values = extractor.extract_point(lat, lon)

    # Base environmental columns
    base_cols = [f"BIO{i}" for i in range(1, 20)] + ["elevation"]

    sample = pd.DataFrame([env_values], columns=base_cols)
    sample = add_engineered_features(sample)

    if bundle["type"] == "catboost":
        sample["species"] = species_name

        # Match CatBoost training columns
        sample = sample[bundle["feature_names"]]

        # Impute numeric only
        sample[bundle["numeric_features"]] = bundle["imputer"].transform(
            sample[bundle["numeric_features"]]
        )

        prob = bundle["model"].predict_proba(sample)[0][1]

    else:
        encoder = bundle["encoder"]

        if species_name not in encoder.classes_:
            return None

        sample["species_id"] = encoder.transform([species_name])[0]

        # Match LightGBM model features
        sample = sample[bundle["feature_names"]]

        sample = pd.DataFrame(
            bundle["imputer"].transform(sample),
            columns=bundle["feature_names"]
        )

        prob = bundle["model"].predict_proba(sample)[0][1]

    return float(prob)


# -----------------------
# 6. Test Samples
# -----------------------
test_samples = [
    ("Beirut", 33.8938, 35.5018, "Cedrus libani"),
    ("Cedars of God", 34.2436, 36.0486, "Cedrus libani"),
    ("Bcharre", 34.2508, 36.0106, "Cedrus libani"),
    ("Byblos", 34.1230, 35.6519, "Ceratonia siliqua"),
    ("Beirut", 33.8938, 35.5018, "Ceratonia siliqua"),
    ("Faraya", 33.9944, 35.8172, "Ceratonia siliqua"),
    ("Cedars of God", 34.2436, 36.0486, "Abies cilicica"),
    ("Faraya", 33.9944, 35.8172, "Abies cilicica"),
    ("Bcharre", 34.2508, 36.0106, "Juniperus excelsa"),
    ("Tyre", 33.2700, 35.2038, "Cedrus libani"),
    ("Byblos", 34.1230, 35.6519, "Quercus coccifera calliprinos"),
    ("Bcharre", 34.2508, 36.0106, "Quercus infectoria boissieri"),
]


# -----------------------
# 7. Compare Models
# -----------------------
results = []

for city, lat, lon, species in test_samples:
    row = {
        "city": city,
        "species": species,
        "lat": lat,
        "lon": lon,
    }

    for bundle in models:
        prob = predict_with_model(bundle, city, lat, lon, species)

        if prob is None:
            row[bundle["name"]] = np.nan
        else:
            row[bundle["name"]] = prob

    results.append(row)


comparison_df = pd.DataFrame(results)

# Add best model per sample
model_cols = list(MODEL_DIRS.keys())

comparison_df["best_model"] = comparison_df[model_cols].idxmax(axis=1)
comparison_df["best_probability"] = comparison_df[model_cols].max(axis=1)
comparison_df["best_class"] = comparison_df["best_probability"].apply(suitability_class)

comparison_df

Loading: CatBoost_v1
Loading: LightGBM_v5
Loading: LightGBM_v4
Loading: LightGBM_v3
Loaded models: ['CatBoost_v1', 'LightGBM_v5', 'LightGBM_v4', 'LightGBM_v3']


,city,species,lat,lon,CatBoost_v1,LightGBM_v5,LightGBM_v4,LightGBM_v3,best_model,best_probability,best_class
0,Beirut,Cedrus libani,33.8938,35.5018,0.001366,0.015594,0.034200,0.059734,LightGBM_v3,0.059734,Low / not suitable
1,Cedars of God,Cedrus libani,34.2436,36.0486,0.540965,0.674516,0.721625,0.586523,LightGBM_v4,0.721625,Suitable
2,Bcharre,Cedrus libani,34.2508,36.0106,0.551180,0.532650,0.566162,0.674556,LightGBM_v3,0.674556,Suitable
3,Byblos,Ceratonia siliqua,34.1230,35.6519,0.926218,0.950904,0.918992,0.985767,LightGBM_v3,0.985767,Highly suitable
4,Beirut,Ceratonia siliqua,33.8938,35.5018,0.789922,0.712382,0.800888,0.832810,LightGBM_v3,0.832810,Highly suitable
5,Faraya,Ceratonia siliqua,33.9944,35.8172,0.002020,0.021000,0.014283,0.024090,LightGBM_v3,0.024090,Low / not suitable
6,Cedars of God,Abies cilicica,34.2436,36.0486,0.371673,0.707631,0.738559,0.527037,LightGBM_v4,0.738559,Suitable
7,Faraya,Abies cilicica,33.9944,35.8172,0.034441,0.014304,0.015813,0.008149,CatBoost_v1,0.034441,Low / not suitable
8,Bcharre,Juniperus excelsa,34.2508,36.0106,0.930415,0.954444,0.970405,0.931928,LightGBM_v4,0.970405,Highly suitable
9,Tyre,Cedrus libani,33.2700,35.2038,0.000120,0.000922,0.000595,0.000640,LightGBM_v5,0.000922,Low / not suitable


In [ ]:
output_path = f"{BASE_DIR}/model_comparison_samples.csv"
comparison_df.to_csv(output_path, index=False)

print("Saved comparison to:", output_path)

Saved comparison to: /content/drive/MyDrive/PhytoLB/model_comparison_samples.csv


In [ ]:
comparison_df[
    ["city", "species"] + list(MODEL_DIRS.keys()) + ["best_model", "best_class"]
]

,city,species,CatBoost_v1,LightGBM_v5,LightGBM_v4,LightGBM_v3,best_model,best_class
0,Beirut,Cedrus libani,0.001366,0.015594,0.034200,0.059734,LightGBM_v3,Low / not suitable
1,Cedars of God,Cedrus libani,0.540965,0.674516,0.721625,0.586523,LightGBM_v4,Suitable
2,Bcharre,Cedrus libani,0.551180,0.532650,0.566162,0.674556,LightGBM_v3,Suitable
3,Byblos,Ceratonia siliqua,0.926218,0.950904,0.918992,0.985767,LightGBM_v3,Highly suitable
4,Beirut,Ceratonia siliqua,0.789922,0.712382,0.800888,0.832810,LightGBM_v3,Highly suitable
5,Faraya,Ceratonia siliqua,0.002020,0.021000,0.014283,0.024090,LightGBM_v3,Low / not suitable
6,Cedars of God,Abies cilicica,0.371673,0.707631,0.738559,0.527037,LightGBM_v4,Suitable
7,Faraya,Abies cilicica,0.034441,0.014304,0.015813,0.008149,CatBoost_v1,Low / not suitable
8,Bcharre,Juniperus excelsa,0.930415,0.954444,0.970405,0.931928,LightGBM_v4,Highly suitable
9,Tyre,Cedrus libani,0.000120,0.000922,0.000595,0.000640,LightGBM_v5,Low / not suitable
